## Import and Load the Train and Test sets

In [65]:
import warnings
warnings.filterwarnings("ignore")    # Ignore all warnings for cleaner script output

# Mount Google Drive (for Colab environment)
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder

# Set file paths to train & test data

train_path = '/content/drive/My Drive/Burnout/new_train.csv'
test_path  = '/content/drive/My Drive/Burnout/Questionnaire_Generated_Data.csv'

# Load train set (Burnout Dataset)
train_df = pd.read_csv(train_path)

# Load test set (actual data from the company - AccessFintech)
test_df = pd.read_csv(test_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
train_df.head(2)

,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate,WFH Setup Available_Yes,Gender_Male,Company Type_Service
0,2.0,3.0,3.8,0.16,False,False,True
1,1.0,2.0,5.0,0.36,True,True,True


In [67]:
test_df.head(2)

,EmployeeID,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,R_MZCoQh8uM5swkkx0,Female,Data,Yes,1,1.0,2.6,0.09
1,R_JNxcv0bxRDLb7uGl,Female,Sales,Yes,2,6.0,3.3,0.28


## Prepare the train set

In [68]:
# Drop 'Company Type' column as it has no explanatory power on the dataset, and also because the test set (company's data) has more than 2 values in this category

if 'Company Type_Service' in train_df.columns:
    train_df = train_df.drop(columns=['Company Type_Service'])

train_df.head(2)

,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate,WFH Setup Available_Yes,Gender_Male
0,2.0,3.0,3.8,0.16,False,False
1,1.0,2.0,5.0,0.36,True,True


## Prepare the test set

In [69]:
# Print top rows of test data for checking
test_df.head(2)

,EmployeeID,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,R_MZCoQh8uM5swkkx0,Female,Data,Yes,1,1.0,2.6,0.09
1,R_JNxcv0bxRDLb7uGl,Female,Sales,Yes,2,6.0,3.3,0.28


In [70]:
# Extract y_test (ground truth) before modifying test DataFrame
y_test = test_df['Burn Rate']

# Drop 'Employee ID', 'Burn Rate', and 'Company Type' to get test features
X_test = test_df.drop(columns=['EmployeeID', 'Burn Rate', 'Company Type'])

# One-hot encode categorical variables in test set only
X_test = pd.get_dummies(
    X_test,
    columns=['WFH Setup Available','Gender'], # 'Company Type' removed from here
    drop_first=True
)

# Print processed test data head
test_df.head(2)

,EmployeeID,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,R_MZCoQh8uM5swkkx0,Female,Data,Yes,1,1.0,2.6,0.09
1,R_JNxcv0bxRDLb7uGl,Female,Sales,Yes,2,6.0,3.3,0.28


## Make the train and test datasets align with each other

In [71]:
# Add any missing columns in test set (relative to train) as zeros, so columns match
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0   # If feature is missing in test, so add column of zeros

# Drop extra columns in X_test that are not in X_train, to force exact column order
X_test = X_test[[col for col in X_train.columns]]

X_train.head(2)
X_test.head(2)

,Designation,Resource Allocation,Mental Fatigue Score,WFH Setup Available_Yes,Gender_Male,Company Type_Service
0,1,1.0,2.6,True,False,0
1,2,6.0,3.3,True,False,0


In [72]:
X_test.head(2)

,Designation,Resource Allocation,Mental Fatigue Score,WFH Setup Available_Yes,Gender_Male,Company Type_Service
0,1,1.0,2.6,True,False,0
1,2,6.0,3.3,True,False,0


## Predict with XGBoost

In [73]:
# XGBoost Hyperparameters: Version 1 (manual/grid search)
params_xgb_1 = {
    'colsample_bytree': 1.0,
    'learning_rate':    0.1,
    'max_depth':        5,
    'n_estimators':     100,
    'subsample':        1.0,
    'objective':        'reg:squarederror',
    'random_state':     42,
    'n_jobs':           -1
}

# XGBoost Hyperparameters: Version 2 (Optuna best)
params_xgb_2 = {
    'n_estimators':     788,
    'learning_rate':    0.006898546839320549,
    'max_depth':        6,
    'subsample':        0.6919002009978714,
    'colsample_bytree': 0.7301631111795839,
    'reg_alpha':        1.1127968534340953e-05,
    'reg_lambda':       0.0002193631522224463,
    'objective':        'reg:squarederror',
    'random_state':     42,
    'n_jobs':           -1
}

# Store evaluation metrics for each model version for easy comparison
results = []

# Loop over both model configurations: manual and Optuna-tuned
for name, params in [
    ("XGBoost", params_xgb_1),
    ("XGBoost_Optuna", params_xgb_2)
]:
    model = XGBRegressor(**params)    # Instantiate XGBRegressor with given hyperparameters
    model.fit(X_train, y_train)       # Fit on full training data
    y_pred = model.predict(X_test)    # Predict on processed test features
    if y_test is not None:
        mae  = mean_absolute_error(y_test, y_pred)          # Compute and store MAE, RMSE, R2 for this model on test set
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)
        results.append({
            "Model": name,
            "MAE":   mae,
            "RMSE":  rmse,
            "R2":    r2
        })
    else:
        print(f"\nPredictions for {name}:")    # If no y_test labels, just print predictions for manual review
        print(y_pred)

# Print comparison table if y_test is available, for clear reporting
if results:
    df_results = pd.DataFrame(results)
    print("\n==== XGBoost Comparison on Test Set ====")
    print(df_results.round(4).to_string(index=False))


==== XGBoost Comparison on Test Set ====
         Model   MAE   RMSE     R2
       XGBoost 0.084 0.1053 0.7160
XGBoost_Optuna 0.082 0.1034 0.7263
